<a href="https://colab.research.google.com/github/Osondu-ifunanya/Evapotranspiration-estimation-using-ML-regression-with-climate-variables/blob/main/Evapotranspiration%20using%20ML%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Evapotranspiration Estimation using Machine Learning Regression
(Synthetic Climate Data)

Workflow:
1. Generate synthetic climate variables
2. Simulate ET using physical relationships
3. Train ML regression models
4. Evaluate model performance
5. Export results
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

np.random.seed(42)

# ---------------------------------------
# 1. Generate Synthetic Climate Data
# ---------------------------------------

n_samples = 1500

temperature = np.random.normal(25, 5, n_samples)      # °C
humidity = np.random.uniform(30, 90, n_samples)       # %
wind_speed = np.random.uniform(0.5, 5, n_samples)     # m/s
solar_radiation = np.random.uniform(10, 30, n_samples) # MJ/m²/day

# ---------------------------------------
# 2. Simulate Evapotranspiration (ET)
# ---------------------------------------

# Simplified ET relationship (synthetic)
et = (
    0.4 * solar_radiation +
    0.3 * temperature -
    0.2 * humidity +
    0.5 * wind_speed
)

# Add noise
et += np.random.normal(0, 1.5, n_samples)

# ---------------------------------------
# 3. Prepare Dataset
# ---------------------------------------

df = pd.DataFrame({
    "temperature": temperature,
    "humidity": humidity,
    "wind_speed": wind_speed,
    "solar_radiation": solar_radiation,
    "ET": et
})

X = df.drop(columns=["ET"])
y = df["ET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# ---------------------------------------
# 4. Train ML Models
# ---------------------------------------

rf = RandomForestRegressor(n_estimators=200, random_state=42)
gb = GradientBoostingRegressor()
lr = LinearRegression()

rf.fit(X_train, y_train)
gb.fit(X_train, y_train)
lr.fit(X_train, y_train)

# ---------------------------------------
# 5. Predictions
# ---------------------------------------

rf_pred = rf.predict(X_test)
gb_pred = gb.predict(X_test)
lr_pred = lr.predict(X_test)

# Ensemble (average)
ensemble_pred = (rf_pred + gb_pred + lr_pred) / 3

# ---------------------------------------
# 6. Evaluation
# ---------------------------------------

def evaluate(y_true, y_pred, name):
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name} -> R²: {r2:.3f}, RMSE: {rmse:.3f}")

print("Model Performance:")
evaluate(y_test, rf_pred, "Random Forest")
evaluate(y_test, gb_pred, "Gradient Boosting")
evaluate(y_test, lr_pred, "Linear Regression")
evaluate(y_test, ensemble_pred, "Ensemble")

# ---------------------------------------
# 7. Visualization
# ---------------------------------------

plt.figure(figsize=(6,5))
plt.scatter(y_test, ensemble_pred, alpha=0.5)
plt.xlabel("Observed ET")
plt.ylabel("Predicted ET")
plt.title("Observed vs Predicted ET (Ensemble)")
plt.grid()
plt.show()

# ---------------------------------------
# 8. Export Results
# ---------------------------------------

output = pd.DataFrame({
    "Observed_ET": y_test.values,
    "RF_Predicted": rf_pred,
    "GB_Predicted": gb_pred,
    "LR_Predicted": lr_pred,
    "Ensemble_ET": ensemble_pred
})

output.to_excel("evapotranspiration_results.xlsx", index=False)

print("Results exported to evapotranspiration_results.xlsx")